# 第11章 指示チューニング

## 11.3 指示チューニングしたモデルの評価

In [1]:
!pip install flexeval bitsandbytes

### 11.3.1 モデルの動作確認

In [2]:
from flexeval import HuggingFaceLM
model_name = "llm-book/Swallow-7b-hf-oasst1-21k-ja"
llm = HuggingFaceLM(model=model_name)

2026-06-01 00:09:01.725 | INFO     | flexeval.core.language_model.hf_lm:__init__:229 - amp_dtype: None
2026-06-01 00:09:01.726 | INFO     | flexeval.core.language_model.hf_lm:__init__:230 - random seed: 42


In [3]:
input_messages = [{"role": "user", "content": "1+1はなんでしょうか？"}]
print(llm.generate_chat_response(input_messages))

`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

2026-06-01 00:09:58.994 | INFO     | flexeval.core.language_model.hf_lm:wrapper:270 - model device: cuda:0
2026-06-01 00:09:58.995 | INFO     | flexeval.core.language_model.hf_lm:wrapper:271 - model dtype: torch.bfloat16


LMOutput(text='1+1は2である。', raw_text=None, reasoning_text=None, finish_reason='stop', tool_calls=None, tool_call_validation_result=None)


### 11.3.2 指示追従性能の評価

In [ ]:
# このセルを実行すると、無料版のColabなどの低メモリ環境下ではRAMが不足しクラッシュする可能性があります
# その場合はランタイムを再起動し、動作確認をスキップして続きのセルを実行してください
import gc
import torch

# GPUに載せたモデルをCPUに移し、GPUを解放する
# HuggingFaceLMは遅延ロードのため、動作確認をしていない場合はllm.modelがNoneになる
if llm.model is not None:
    llm.model.cpu()
del llm
gc.collect()
torch.cuda.empty_cache()

In [5]:
from google.colab import drive

# Googleドライブを"drive"ディレクトリ以下にマウント
drive.mount("drive")

Drive already mounted at drive; to attempt to forcibly remount, call drive.mount("drive", force_remount=True).


In [6]:
# 無料版のT4 GPUなど、低メモリ環境での評価コマンド
# 量子化とバッチサイズを小さく設定
!flexeval_lm \
  --language_model HuggingFaceLM \
  --language_model.model "llm-book/Swallow-7b-hf-oasst1-21k-ja" \
  --language_model.model_kwargs.load_in_4bit true \
  --eval_setup "vicuna-ja" \
  --eval_setup.gen_kwargs '{do_sample: True, temperature: 0.7, top_p: 0.9, max_new_tokens: 1024}' \
  --eval_setup.batch_size 1 \
  --save_dir "./drive/MyDrive/llm-book/IT_eval/vicuna-ja" \
  --force true

2026-06-01 00:10:23.888 | INFO     | flexeval.utils.module_utils:__call__:83 - Resolved config name 'vicuna-ja' to path '/usr/local/lib/python3.12/dist-packages/flexeval/preset_configs/EvalSetup/ja_chat/vicuna-ja.jsonnet'
2026-06-01 00:10:24.856 | INFO     | flexeval.scripts.flexeval_lm:main:222 - Namespace(language_model=Namespace(class_path='flexeval.HuggingFaceLM', init_args=Namespace(model='llm-book/Swallow-7b-hf-oasst1-21k-ja', model_kwargs={'load_in_4bit': True}, tokenizer=None, tokenizer_kwargs=None, add_special_tokens=False, amp_dtype=None, random_seed=42, load_peft=False, custom_chat_template=None, chat_template_kwargs=None, system_message=None, default_gen_kwargs=None, string_processors=None, model_limit_tokens='default', tool_parser=None, tools=None)), eval_setup=Namespace(class_path='flexeval.ChatResponse', init_args=Namespace(eval_dataset=Namespace(class_path='flexeval.ChatbotBench', init_args=Namespace(path_or_name='vicuna-ja', ref_path_or_name='vicuna-ja-ref-gpt4', need_

In [7]:
import json
from pathlib import Path

save_dir = "./drive/MyDrive/llm-book/IT_eval/vicuna-ja"

with open(Path(save_dir) / "outputs.jsonl") as f:
    for line in f:
        item = json.loads(line)
        print("===== 入力 ====")
        print(item["task_inputs"]["messages"][0]["content"])
        print("===== モデル出力 ====")
        print(item["lm_output"])
        break   # 全件確認する場合は消してください

FileNotFoundError: [Errno 2] No such file or directory: 'drive/MyDrive/llm-book/IT_eval/vicuna-ja/outputs.jsonl'

In [ ]:
!flexeval_presets assistant_eval_ja_single_turn

In [ ]:
import nest_asyncio
from flexeval import instantiate_from_config

# assistant_eval_ja_single_turnの設定ファイルからMetricをインスタンス化
